# Phase 2 Validation — Frame Extraction + YOLO + Annotation

Interactive notebook for validating each phase checkpoint.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FRAMES_DIR = PROJECT_ROOT / "data" / "frames"
HIGHLIGHTS_DIR = PROJECT_ROOT / "data" / "highlights"

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from manifest import load_manifest

print(f"Project root: {PROJECT_ROOT}")
print(f"Frames dir: {FRAMES_DIR}")

# List extracted matches
extracted = sorted([d.name for d in FRAMES_DIR.iterdir() if d.is_dir()]) if FRAMES_DIR.exists() else []
print(f"Extracted matches: {len(extracted)}")
for m in extracted[:10]:
    print(f"  {m}")

## 2A Validation: Frame Extraction

Check that keyframes are tactical wide shots and sequences are continuous.

In [ ]:
# Pick a test match
match_id = extracted[0] if extracted else "NO_MATCHES_EXTRACTED"
match_dir = FRAMES_DIR / match_id
print(f"Inspecting: {match_id}")

# Load extraction metadata
with open(match_dir / "extraction.json") as f:
    meta = json.load(f)

print(f"Keyframes: {len(meta['keyframes'])}")
print(f"Sequences: {len(meta['sequences'])}")
for seq in meta['sequences']:
    print(f"  {seq['dir']}: {seq['frame_count']} frames, {seq['duration']:.1f}s")

In [ ]:
# Display keyframes in a grid
keyframes_dir = match_dir / "keyframes"
keyframe_files = sorted(keyframes_dir.glob("*.jpg"))

cols = 4
rows = (len(keyframe_files) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten() if rows > 1 else [axes] if rows == 1 and cols == 1 else axes.flatten()

for i, ax in enumerate(axes):
    if i < len(keyframe_files):
        img = cv2.imread(str(keyframe_files[i]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(keyframe_files[i].name, fontsize=9)
    ax.axis('off')

plt.suptitle(f"Keyframes: {match_id}", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Display first and last frames of a sequence (verify continuity)
if meta['sequences']:
    seq_name = meta['sequences'][0]['dir']
    seq_dir = match_dir / "sequences" / seq_name
    seq_frames = sorted(seq_dir.glob("*.jpg"))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, idx in enumerate([0, len(seq_frames)//2, -1]):
        img = cv2.imread(str(seq_frames[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].set_title(f"{seq_frames[idx].name}")
        axes[i].axis('off')
    
    plt.suptitle(f"Sequence {seq_name} — first / middle / last (should be continuous, no camera cut)", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No sequences extracted")

## 2B Validation: YOLO Detections

_(Run after Milestone 2)_

In [ ]:
# Load detections
det_path = match_dir / "detections.json"
assert det_path.exists(), f"Run detect_players.py first: {det_path}"

with open(det_path) as f:
    detections = json.load(f)

# Player count per keyframe
print("Player detections per keyframe:")
for fname, det in detections.get("keyframes", {}).items():
    n_players = len(det.get("players", []))
    has_ball = "ball" in det and det["ball"] is not None
    print(f"  {fname}: {n_players} players, ball={'YES' if has_ball else 'NO'}")

In [ ]:
# Overlay bboxes on a keyframe
sample_frame = list(detections["keyframes"].keys())[0]
img = cv2.imread(str(keyframes_dir / sample_frame))
det = detections["keyframes"][sample_frame]

for player in det.get("players", []):
    x1, y1, x2, y2 = [int(v) for v in player["bbox"]]
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

if det.get("ball"):
    bx1, by1, bx2, by2 = [int(v) for v in det["ball"]["bbox"]]
    cv2.circle(img, ((bx1+bx2)//2, (by1+by2)//2), 10, (0, 0, 255), -1)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Detections: {sample_frame} ({len(det.get('players',[]))} players)")
plt.axis('off')
plt.show()

In [ ]:
# Track consistency in a sequence
if detections.get("sequences"):
    seq_key = list(detections["sequences"].keys())[0]
    tracks = detections["sequences"][seq_key].get("tracks", {})
    print(f"Sequence {seq_key}: {len(tracks)} unique track IDs")
    
    # Plot trajectory of top 5 tracks
    fig, ax = plt.subplots(figsize=(10, 6))
    for tid, positions in list(tracks.items())[:5]:
        xs = [(p["bbox"][0] + p["bbox"][2]) / 2 for p in positions]
        ys = [(p["bbox"][1] + p["bbox"][3]) / 2 for p in positions]
        ax.plot(xs, ys, '-o', markersize=3, label=f"Track {tid}")
    
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.invert_yaxis()
    ax.legend()
    ax.set_title(f"Player trajectories — {seq_key}")
    plt.show()

## 2C Validation: Annotations + Metrics

_(Run after Milestone 3)_

In [ ]:
# Display annotated frames
annotated_dir = match_dir / "annotated"
assert annotated_dir.exists(), f"Run annotate_frames.py first"

ann_files = sorted(annotated_dir.glob("*.jpg"))[:4]
fig, axes = plt.subplots(1, len(ann_files), figsize=(5 * len(ann_files), 5))
if len(ann_files) == 1:
    axes = [axes]

for ax, f in zip(axes, ann_files):
    img = cv2.imread(str(f))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f.name, fontsize=9)
    ax.axis('off')

plt.suptitle("Annotated frames — team colors, defensive lines, compactness")
plt.tight_layout()
plt.show()

In [ ]:
# Load and display temporal metrics
metrics_path = match_dir / "metrics.json"
assert metrics_path.exists(), "Run annotate_frames.py first"

with open(metrics_path) as f:
    metrics = json.load(f)

print("=== Temporal Metrics ===")
print(f"\nAggregated:")
for k, v in metrics.get("aggregated", {}).items():
    print(f"  {k}: {v:.3f}")

print(f"\nPer-sequence:")
for seq_name, vals in metrics.get("sequences", {}).items():
    print(f"  {seq_name}:")
    for k, v in vals.items():
        flag = " ⚠️" if (v != v or abs(v) > 100) else ""  # NaN or outlier
        print(f"    {k}: {v:.3f}{flag}")

# Sanity checks
issues = []
for seq_name, vals in metrics.get("sequences", {}).items():
    for k, v in vals.items():
        if v != v:  # NaN
            issues.append(f"{seq_name}/{k} is NaN")
        elif abs(v) > 100:
            issues.append(f"{seq_name}/{k} = {v:.1f} (outlier?)")

if issues:
    print(f"\n⚠️ {len(issues)} potential issues:")
    for i in issues:
        print(f"  - {i}")
else:
    print("\n✓ All metrics look reasonable")

## Summary — Pass/Fail Checklist

In [ ]:
print("Phase 2 Validation Checklist")
print("=" * 40)

checks = {
    "2A: Keyframes extracted": len(meta.get('keyframes', [])) > 0,
    "2A: Sequences extracted": len(meta.get('sequences', [])) > 0,
    "2A: Keyframes are tactical (visual check)": None,  # manual
    "2A: Sequences are continuous (visual check)": None,  # manual
    "2B: detections.json exists": det_path.exists(),
    "2B: 6-30 players per keyframe": all(
        6 <= len(d.get('players', [])) <= 30
        for d in detections.get('keyframes', {}).values()
    ) if detections.get('keyframes') else False,
    "2C: Annotated frames exist": annotated_dir.exists() if 'annotated_dir' in dir() else False,
    "2C: Metrics have no NaN": len(issues) == 0 if 'issues' in dir() else False,
}

for check, result in checks.items():
    if result is None:
        status = "👁️ MANUAL"
    elif result:
        status = "✓ PASS"
    else:
        status = "✗ FAIL"
    print(f"  [{status}] {check}")